In [1]:
# ============================================================
# 09_trade_candidate_pricing_v0_1
#
# Goal:
#   Price mapped naked short put/call trade candidates from
#   notebook 08.
#
# Scope:
#   Entry-to-expiry only.
#   No daily MTM.
#   No risk sizing.
#   No portfolio simulation.
#
# Entry convention:
#   Sell short option at bid.
#
# Expiry convention:
#   Use SPX close on selected_expiry_date.
#
# Normalization:
#   Puts scaled to same put notional.
#   Calls scaled to same call notional.
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RAW_DATA_DIR = DATA_DIR / "raw"
AUDIT_DIR = DATA_DIR / "audit"

TRADE_CANDIDATE_PANEL_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.parquet"
VRP_PANEL_PARQUET = PROCESSED_DATA_DIR / "vrp_panel_v0_1.parquet"

PRICING_OUTPUT_CSV = PROCESSED_DATA_DIR / "trade_candidate_pricing_v0_1.csv"
PRICING_OUTPUT_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_pricing_v0_1.parquet"

PRICING_AUDIT_CSV = AUDIT_DIR / "trade_candidate_pricing_audit_v0_1.csv"

NORMALIZED_PUT_NOTIONAL = 1_000_000.0
NORMALIZED_CALL_NOTIONAL = 1_000_000.0
SPX_CONTRACT_MULTIPLIER = 100.0

print("Project root:", PROJECT_ROOT)
print("Trade candidate file exists:", TRADE_CANDIDATE_PANEL_PARQUET.exists())
print("VRP panel exists:", VRP_PANEL_PARQUET.exists())
print("Put normalized notional:", NORMALIZED_PUT_NOTIONAL)
print("Call normalized notional:", NORMALIZED_CALL_NOTIONAL)

Project root: C:\Users\patri\vrp_project
Trade candidate file exists: True
VRP panel exists: True
Put normalized notional: 1000000.0
Call normalized notional: 1000000.0


In [2]:
# ============================================================
# Load mapped trade candidate panel from notebook 08
# ============================================================

candidate_df = pd.read_parquet(TRADE_CANDIDATE_PANEL_PARQUET).copy()

required_candidate_cols = [
    "trade_date",
    "trade_side",
    "option_right",
    "option_structure",
    "selected_expiry_date",
    "actual_dte",
    "selected_chain_file",
    "short_strike_selected",
    "spx_close",
    "mapping_status",
]

missing_candidate_cols = [
    c for c in required_candidate_cols
    if c not in candidate_df.columns
]

if missing_candidate_cols:
    raise ValueError(f"Missing candidate columns: {missing_candidate_cols}")

candidate_df = candidate_df[candidate_df["mapping_status"] == "mapped"].copy()

candidate_df["trade_date"] = candidate_df["trade_date"].astype(int)
candidate_df["selected_expiry_date"] = candidate_df["selected_expiry_date"].astype(int)
candidate_df["short_strike_selected"] = candidate_df["short_strike_selected"].astype(float)
candidate_df["spx_close"] = candidate_df["spx_close"].astype(float)

print("Mapped candidate rows:", len(candidate_df))
print("Unique trade dates:", candidate_df["trade_date"].nunique())

print("\nCandidates by side:")
display(candidate_df["trade_side"].value_counts().rename("count").to_frame())

print("\nCandidate expiry range:")
print(candidate_df["selected_expiry_date"].min(), "to", candidate_df["selected_expiry_date"].max())

display(candidate_df.tail(20))

Mapped candidate rows: 1829
Unique trade dates: 1245

Candidates by side:


,count
trade_side,
put,1080
call,749



Candidate expiry range:
20180907 to 20260702


,trade_date,underlyer,trade_side,option_structure,option_right,selected_tenor,target_expiry_date,expiry_selection_rule,exit_rule,short_strike_rule,...,expiry_mapping_status,actual_dte,short_strike_raw_actual_dte,short_strike_selected,strike_mapping_status,short_strike_raw_target_tenor,mapping_status,dte_minus_selected_tenor,short_strike_moneyness,short_strike_pct_otm
1809,20260520,SPX,call,naked_short_call,C,33.0,20260622,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,mapped,37.0,8265.191557,8300.0,mapped,8218.920273,mapped,4.0,1.116647,0.116647
1810,20260526,SPX,put,naked_short_put,P,18.0,20260613,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,mapped,17.0,7519.000000,7515.0,mapped,7519.000000,mapped,-1.0,0.999452,0.000548
1811,20260526,SPX,call,naked_short_call,C,18.0,20260613,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,mapped,17.0,8032.826102,8050.0,mapped,8047.719218,mapped,-1.0,1.070604,0.070604
1812,20260527,SPX,put,naked_short_put,P,12.0,20260608,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,mapped,9.0,7520.000000,7520.0,mapped,7520.000000,mapped,-3.0,0.999952,0.000048
1813,20260527,SPX,call,naked_short_call,C,12.0,20260608,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,mapped,9.0,7852.881794,7860.0,mapped,7904.323095,mapped,-3.0,1.045163,0.045163
1814,20260528,SPX,put,naked_short_put,P,12.0,20260609,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,mapped,15.0,7563.000000,7560.0,mapped,7563.000000,mapped,3.0,0.999520,0.000480
1815,20260528,SPX,call,naked_short_call,C,12.0,20260609,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,mapped,15.0,7973.512033,7975.0,mapped,7930.239636,mapped,3.0,1.054388,0.054388
1816,20260529,SPX,put,naked_short_put,P,21.0,20260619,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,mapped,20.0,7580.000000,7580.0,mapped,7580.000000,mapped,-1.0,0.999992,0.000008
1817,20260529,SPX,call,naked_short_call,C,9.0,20260607,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,mapped,7.0,7846.198069,7850.0,mapped,7881.832204,mapped,-2.0,1.035612,0.035612
1818,20260601,SPX,put,naked_short_put,P,15.0,20260616,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,mapped,17.0,7599.000000,7595.0,mapped,7599.000000,mapped,2.0,0.999347,0.000653


In [3]:
# ============================================================
# Load SPX close history
# ============================================================

vrp_panel_df = pd.read_parquet(VRP_PANEL_PARQUET).copy()

required_spx_cols = ["trade_date", "spx_close"]
missing_spx_cols = [
    c for c in required_spx_cols
    if c not in vrp_panel_df.columns
]

if missing_spx_cols:
    raise ValueError(f"Missing SPX close columns: {missing_spx_cols}")

spx_close_df = (
    vrp_panel_df[["trade_date", "spx_close"]]
    .drop_duplicates()
    .copy()
)

spx_close_df["trade_date"] = spx_close_df["trade_date"].astype(int)
spx_close_df["spx_close"] = spx_close_df["spx_close"].astype(float)

# Rename for expiry merge
expiry_spx_df = spx_close_df.rename(
    columns={
        "trade_date": "selected_expiry_date",
        "spx_close": "expiry_spx_close",
    }
)

print("SPX close rows:", len(spx_close_df))
print("SPX close date range:", spx_close_df["trade_date"].min(), "to", spx_close_df["trade_date"].max())

print("\nCandidate expiry max:", candidate_df["selected_expiry_date"].max())
print("SPX close max:", spx_close_df["trade_date"].max())

missing_expiry_dates = sorted(
    set(candidate_df["selected_expiry_date"].unique())
    - set(expiry_spx_df["selected_expiry_date"].unique())
)

print("\nMissing expiry close dates:", len(missing_expiry_dates))
print(missing_expiry_dates[:20])

SPX close rows: 2011
SPX close date range: 20180625 to 20260625

Candidate expiry max: 20260702
SPX close max: 20260625

Missing expiry close dates: 2
[np.int64(20260626), np.int64(20260702)]


In [4]:
# ============================================================
# Cached ThetaData chain reader
# ============================================================

CHAIN_DF_CACHE = {}

def convert_pickle_object_to_dataframe(obj):
    """
    Convert pickle-loaded object to DataFrame when possible.
    """
    if isinstance(obj, pd.DataFrame):
        return obj

    if isinstance(obj, dict):
        dataframe_items = {
            k: v for k, v in obj.items()
            if isinstance(v, pd.DataFrame)
        }

        if len(dataframe_items) > 0:
            largest_key = max(dataframe_items, key=lambda k: len(dataframe_items[k]))
            return dataframe_items[largest_key]

        try:
            return pd.DataFrame(obj)
        except Exception as e:
            raise ValueError(f"Pickle dict could not be converted to DataFrame: {e}")

    if isinstance(obj, list):
        try:
            return pd.DataFrame(obj)
        except Exception as e:
            raise ValueError(f"Pickle list could not be converted to DataFrame: {e}")

    raise ValueError(f"Unsupported pickle object type: {type(obj)}")


def read_cached_chain_file(file_path):
    """
    Read local cached ThetaData chain file.
    Supports .pkl, .parquet, and .csv.
    """
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".pkl":
        obj = pd.read_pickle(file_path)
        df = convert_pickle_object_to_dataframe(obj)

    elif suffix == ".parquet":
        df = pd.read_parquet(file_path)

    elif suffix == ".csv":
        df = pd.read_csv(file_path)

    else:
        raise ValueError(f"Unsupported cached chain file suffix: {suffix}")

    return df


def get_cached_chain_df(file_path):
    """
    Read a chain once and cache it in memory.
    """
    file_path_str = str(file_path)

    if file_path_str in CHAIN_DF_CACHE:
        return CHAIN_DF_CACHE[file_path_str]

    df = read_cached_chain_file(file_path_str).copy()

    CHAIN_DF_CACHE[file_path_str] = df

    return df


# Quick read test
sample_chain_file = candidate_df["selected_chain_file"].iloc[0]
sample_chain_df = get_cached_chain_df(sample_chain_file)

print("Sample chain file:", sample_chain_file)
print("Sample chain rows:", len(sample_chain_df))
print("Sample columns:", list(sample_chain_df.columns))

display(sample_chain_df.head())

Sample chain file: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20180823_20180921_160000.pkl
Sample chain rows: 540
Sample columns: ['root', 'expiration', 'strike', 'right', 'bid', 'ask', 'mid', 'bid_size', 'ask_size', 'bid_exchange', 'ask_exchange', 'bid_condition', 'ask_condition', 'timestamp']


,root,expiration,strike,right,bid,ask,mid,bid_size,ask_size,bid_exchange,ask_exchange,bid_condition,ask_condition,timestamp
0,SPX,20180921,2645.0,C,215.30,217.60,216.45,120,120,5,5,50,50,2018-08-23T16:00:00.000
1,SPX,20180921,2645.0,P,3.60,3.80,3.70,1288,366,5,5,50,50,2018-08-23T16:00:00.000
2,SPX,20180921,1920.0,P,0.25,0.35,0.30,600,1280,5,5,50,50,2018-08-23T16:00:00.000
3,SPX,20180921,1920.0,C,935.30,937.70,936.50,120,120,5,5,50,50,2018-08-23T16:00:00.000
4,SPX,20180921,2560.0,P,2.35,2.45,2.40,324,873,5,5,50,50,2018-08-23T16:00:00.000


In [5]:
# ============================================================
# Entry quote lookup
# ============================================================

def lookup_entry_quote_for_candidate(row):
    """
    Find bid/ask/mid for the exact selected option in the cached
    close chain.

    Entry convention:
        Sell at bid.
    """
    file_path = row["selected_chain_file"]
    option_right = str(row["option_right"]).upper()
    selected_expiry = int(row["selected_expiry_date"])
    selected_strike = float(row["short_strike_selected"])

    try:
        chain_df = get_cached_chain_df(file_path).copy()
    except Exception as e:
        return {
            "entry_bid": np.nan,
            "entry_ask": np.nan,
            "entry_mid": np.nan,
            "entry_spread": np.nan,
            "entry_spread_pct_mid": np.nan,
            "entry_quote_status": f"chain_read_error: {e}",
        }

    required_cols = ["expiration", "right", "strike", "bid", "ask"]
    missing_cols = [c for c in required_cols if c not in chain_df.columns]

    if missing_cols:
        return {
            "entry_bid": np.nan,
            "entry_ask": np.nan,
            "entry_mid": np.nan,
            "entry_spread": np.nan,
            "entry_spread_pct_mid": np.nan,
            "entry_quote_status": f"missing_chain_columns: {missing_cols}",
        }

    chain_df["expiration"] = pd.to_numeric(chain_df["expiration"], errors="coerce")
    chain_df["strike"] = pd.to_numeric(chain_df["strike"], errors="coerce")
    chain_df["bid"] = pd.to_numeric(chain_df["bid"], errors="coerce")
    chain_df["ask"] = pd.to_numeric(chain_df["ask"], errors="coerce")
    chain_df["right"] = chain_df["right"].astype(str).str.upper()

    option_rows = chain_df[
        (chain_df["expiration"] == selected_expiry)
        & (chain_df["right"] == option_right)
        & (np.isclose(chain_df["strike"], selected_strike))
    ].copy()

    if option_rows.empty:
        return {
            "entry_bid": np.nan,
            "entry_ask": np.nan,
            "entry_mid": np.nan,
            "entry_spread": np.nan,
            "entry_spread_pct_mid": np.nan,
            "entry_quote_status": "selected_option_not_found_in_chain",
        }

    # If duplicates exist, use the row with the tightest nonnegative spread.
    option_rows["spread"] = option_rows["ask"] - option_rows["bid"]
    option_rows = option_rows.sort_values("spread", ascending=True)

    selected = option_rows.iloc[0]

    bid = selected["bid"]
    ask = selected["ask"]
    mid = (bid + ask) / 2.0
    spread = ask - bid

    if pd.isna(mid) or mid <= 0:
        spread_pct_mid = np.nan
    else:
        spread_pct_mid = spread / mid

    status = "mapped"

    if pd.isna(bid) or pd.isna(ask):
        status = "missing_bid_or_ask"
    elif bid < 0 or ask < 0:
        status = "negative_bid_or_ask"
    elif ask < bid:
        status = "crossed_market"
    elif bid == 0:
        status = "zero_bid"

    return {
        "entry_bid": bid,
        "entry_ask": ask,
        "entry_mid": mid,
        "entry_spread": spread,
        "entry_spread_pct_mid": spread_pct_mid,
        "entry_quote_status": status,
    }

In [6]:
# ============================================================
# Apply entry quote lookup
# ============================================================

entry_quote_rows = []

for i, row in candidate_df.iterrows():
    if len(entry_quote_rows) % 250 == 0:
        print(f"Processing entry quotes {len(entry_quote_rows)}/{len(candidate_df)}")

    entry_quote_rows.append(
        lookup_entry_quote_for_candidate(row)
    )

entry_quote_df = pd.DataFrame(entry_quote_rows)

pricing_df = pd.concat(
    [
        candidate_df.reset_index(drop=True),
        entry_quote_df.reset_index(drop=True),
    ],
    axis=1,
)

print("Rows:", len(pricing_df))

print("\nEntry quote status:")
display(
    pricing_df["entry_quote_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nEntry bid by side:")
display(
    pricing_df
    .groupby("trade_side")["entry_bid"]
    .describe()
)

print("\nEntry spread pct mid by side:")
display(
    pricing_df
    .groupby("trade_side")["entry_spread_pct_mid"]
    .describe()
)

display(
    pricing_df[
        [
            "trade_date",
            "trade_side",
            "selected_expiry_date",
            "actual_dte",
            "spx_close",
            "short_strike_selected",
            "entry_bid",
            "entry_ask",
            "entry_mid",
            "entry_spread",
            "entry_quote_status",
        ]
    ].tail(30)
)

Processing entry quotes 0/1829
Processing entry quotes 250/1829
Processing entry quotes 500/1829
Processing entry quotes 750/1829
Processing entry quotes 1000/1829
Processing entry quotes 1250/1829
Processing entry quotes 1500/1829
Processing entry quotes 1750/1829
Rows: 1829

Entry quote status:


,count
entry_quote_status,
mapped,1811
zero_bid,18



Entry bid by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,749.0,0.275968,0.180527,0.0,0.15,0.25,0.400,1.05
put,1080.0,62.152130,28.028062,0.0,41.40,57.80,77.225,214.20



Entry spread pct mid by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,749.0,0.514343,0.352217,0.060606,0.260870,0.400000,0.666667,2.0
put,1079.0,0.056560,0.096916,0.004520,0.013899,0.030164,0.074262,2.0


,trade_date,trade_side,selected_expiry_date,actual_dte,spx_close,short_strike_selected,entry_bid,entry_ask,entry_mid,entry_spread,entry_quote_status
1799,20260513,put,20260605,23.0,7444.25,7440.0,100.40,101.20,100.800,0.80,mapped
1800,20260513,call,20260605,23.0,7444.25,8075.0,0.80,0.95,0.875,0.15,mapped
1801,20260514,call,20260612,29.0,7501.24,8300.0,0.60,0.80,0.700,0.20,mapped
1802,20260515,put,20260612,28.0,7408.50,7405.0,106.90,114.10,110.500,7.20,mapped
1803,20260515,call,20260612,28.0,7408.50,8200.0,0.40,0.55,0.475,0.15,mapped
1804,20260518,put,20260618,31.0,7403.05,7400.0,115.30,116.50,115.900,1.20,mapped
1805,20260518,call,20260618,31.0,7403.05,8170.0,0.40,0.70,0.550,0.30,mapped
1806,20260519,put,20260618,30.0,7353.61,7350.0,115.60,116.50,116.050,0.90,mapped
1807,20260519,call,20260618,30.0,7353.61,8110.0,0.45,0.65,0.550,0.20,mapped
1808,20260520,put,20260626,37.0,7432.97,7430.0,127.80,128.90,128.350,1.10,mapped


In [7]:
# ============================================================
# Merge expiry SPX close and calculate expiry intrinsic value
# ============================================================

pricing_df = pricing_df.merge(
    expiry_spx_df,
    on="selected_expiry_date",
    how="left",
)

pricing_df["expiry_spx_status"] = np.where(
    pricing_df["expiry_spx_close"].notna(),
    "mapped",
    "missing_expiry_spx_close",
)

def calculate_expiry_intrinsic_value(row):
    if pd.isna(row["expiry_spx_close"]):
        return np.nan

    strike = float(row["short_strike_selected"])
    expiry_spx = float(row["expiry_spx_close"])

    if row["trade_side"] == "put":
        return max(strike - expiry_spx, 0.0)

    if row["trade_side"] == "call":
        return max(expiry_spx - strike, 0.0)

    return np.nan


pricing_df["expiry_intrinsic_value"] = pricing_df.apply(
    calculate_expiry_intrinsic_value,
    axis=1,
)

pricing_df["short_option_expired_otm"] = np.where(
    pricing_df["expiry_intrinsic_value"].notna(),
    pricing_df["expiry_intrinsic_value"] == 0.0,
    np.nan,
)

print("Expiry SPX status:")
display(
    pricing_df["expiry_spx_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nExpiry intrinsic value by side:")
display(
    pricing_df
    .groupby("trade_side")["expiry_intrinsic_value"]
    .describe()
)

print("\nExpired OTM rate by side:")
display(
    pricing_df
    .dropna(subset=["short_option_expired_otm"])
    .groupby("trade_side")["short_option_expired_otm"]
    .mean()
    .rename("expired_otm_rate")
    .to_frame()
)

Expiry SPX status:


,count
expiry_spx_status,
mapped,1824
missing_expiry_spx_close,5



Expiry intrinsic value by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,748.0,0.00000,0.000000,0.0,0.0,0.0,0.0000,0.00
put,1076.0,41.46816,93.898009,0.0,0.0,0.0,37.2475,770.92



Expired OTM rate by side:


,expired_otm_rate
trade_side,
call,1.000000
put,0.666357


In [8]:
# ============================================================
# Calculate naked short option P&L
# ============================================================

# Entry convention: sell at bid
pricing_df["entry_credit_points"] = pricing_df["entry_bid"]

pricing_df["short_option_pnl_points"] = (
    pricing_df["entry_credit_points"]
    - pricing_df["expiry_intrinsic_value"]
)

pricing_df["short_option_pnl_dollars_per_contract"] = (
    pricing_df["short_option_pnl_points"]
    * SPX_CONTRACT_MULTIPLIER
)

pricing_df["short_option_win"] = np.where(
    pricing_df["short_option_pnl_points"].notna(),
    pricing_df["short_option_pnl_points"] > 0.0,
    np.nan,
)

# Same-notional normalization.
# This is not risk sizing. It is only apples-to-apples comparison.
pricing_df["normalized_notional"] = np.where(
    pricing_df["trade_side"] == "put",
    NORMALIZED_PUT_NOTIONAL,
    NORMALIZED_CALL_NOTIONAL,
)

pricing_df["normalized_contracts"] = (
    pricing_df["normalized_notional"]
    / (pricing_df["spx_close"] * SPX_CONTRACT_MULTIPLIER)
)

pricing_df["normalized_pnl_dollars"] = (
    pricing_df["short_option_pnl_points"]
    * SPX_CONTRACT_MULTIPLIER
    * pricing_df["normalized_contracts"]
)

pricing_df["normalized_pnl_pct_notional"] = (
    pricing_df["normalized_pnl_dollars"]
    / pricing_df["normalized_notional"]
)

# Equivalent check:
pricing_df["normalized_pnl_pct_notional_check"] = (
    pricing_df["short_option_pnl_points"]
    / pricing_df["spx_close"]
)

pricing_df["normalization_check_diff"] = (
    pricing_df["normalized_pnl_pct_notional"]
    - pricing_df["normalized_pnl_pct_notional_check"]
)

print("Max normalization check diff:")
print(pricing_df["normalization_check_diff"].abs().max())

print("\nWin rate by side:")
display(
    pricing_df
    .dropna(subset=["short_option_win"])
    .groupby("trade_side")["short_option_win"]
    .mean()
    .rename("win_rate")
    .to_frame()
)

print("\nP&L points by side:")
display(
    pricing_df
    .groupby("trade_side")["short_option_pnl_points"]
    .describe()
)

print("\nNormalized P&L pct notional by side:")
display(
    pricing_df
    .groupby("trade_side")["normalized_pnl_pct_notional"]
    .describe()
)

print("\nNormalized P&L dollars by side:")
display(
    pricing_df
    .groupby("trade_side")["normalized_pnl_dollars"]
    .describe()
)

Max normalization check diff:
2.7755575615628914e-17

Win rate by side:


,win_rate
trade_side,
call,0.978610
put,0.775093



P&L points by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,748.0,0.275401,0.17998,0.00,0.15,0.25,0.3625,1.05
put,1076.0,20.470502,95.88457,-657.02,13.49,45.65,69.9000,214.20



Normalized P&L pct notional by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,748.0,0.000062,0.000039,0.000000,0.000033,0.000054,0.000084,0.000282
put,1076.0,0.004493,0.021443,-0.218359,0.003093,0.010684,0.014819,0.042988



Normalized P&L dollars by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,748.0,62.004756,39.314871,0.000000,33.269208,54.128601,84.125500,281.784510
put,1076.0,4492.524298,21442.626055,-218358.822295,3092.756494,10684.299623,14819.007557,42988.137121


In [9]:
# ============================================================
# Pricing status and audit
# ============================================================

pricing_df["pricing_status"] = np.where(
    (pricing_df["entry_quote_status"] == "mapped")
    & (pricing_df["expiry_spx_status"] == "mapped")
    & (pricing_df["short_option_pnl_points"].notna()),
    "priced",
    "not_priced",
)

print("Pricing status:")
display(
    pricing_df["pricing_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nPricing status by side:")
display(
    pricing_df
    .groupby(["trade_side", "pricing_status"])
    .size()
    .unstack(fill_value=0)
)

pricing_audit_df = (
    pricing_df
    .groupby(
        [
            "trade_side",
            "entry_quote_status",
            "expiry_spx_status",
            "pricing_status",
        ]
    )
    .size()
    .reset_index(name="count")
)

display(pricing_audit_df)

priced_df = pricing_df[pricing_df["pricing_status"] == "priced"].copy()

summary_by_side_df = (
    priced_df
    .groupby("trade_side")
    .agg(
        rows=("trade_date", "count"),
        win_rate=("short_option_win", "mean"),
        expired_otm_rate=("short_option_expired_otm", "mean"),
        avg_entry_credit_points=("entry_credit_points", "mean"),
        median_entry_credit_points=("entry_credit_points", "median"),
        avg_pnl_points=("short_option_pnl_points", "mean"),
        median_pnl_points=("short_option_pnl_points", "median"),
        avg_pnl_dollars_per_contract=("short_option_pnl_dollars_per_contract", "mean"),
        avg_normalized_pnl_dollars=("normalized_pnl_dollars", "mean"),
        median_normalized_pnl_dollars=("normalized_pnl_dollars", "median"),
        avg_normalized_pnl_pct_notional=("normalized_pnl_pct_notional", "mean"),
        median_normalized_pnl_pct_notional=("normalized_pnl_pct_notional", "median"),
    )
    .reset_index()
)

print("\nSummary by side:")
display(summary_by_side_df)

Pricing status:


,count
pricing_status,
priced,1806
not_priced,23



Pricing status by side:


pricing_status,not_priced,priced
trade_side,,
call,17,732
put,6,1074


,trade_side,entry_quote_status,expiry_spx_status,pricing_status,count
0,call,mapped,mapped,priced,732
1,call,mapped,missing_expiry_spx_close,not_priced,1
2,call,zero_bid,mapped,not_priced,16
3,put,mapped,mapped,priced,1074
4,put,mapped,missing_expiry_spx_close,not_priced,4
5,put,zero_bid,mapped,not_priced,2



Summary by side:


,trade_side,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,avg_pnl_dollars_per_contract,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,avg_normalized_pnl_pct_notional,median_normalized_pnl_pct_notional
0,call,732,1.000000,1.000000,0.281421,0.25,0.281421,0.25,28.142077,63.360051,56.552231,0.000063,0.000057
1,put,1074,0.776536,0.666667,62.054004,57.80,21.109255,45.75,2110.925512,4704.203880,10689.293638,0.004704,0.010689


In [10]:
# ============================================================
# Save pricing output and audit files
# ============================================================

pricing_df.to_csv(PRICING_OUTPUT_CSV, index=False)
pricing_df.to_parquet(PRICING_OUTPUT_PARQUET, index=False)

pricing_audit_df.to_csv(PRICING_AUDIT_CSV, index=False)

SUMMARY_BY_SIDE_CSV = AUDIT_DIR / "trade_candidate_pricing_summary_by_side_v0_1.csv"
summary_by_side_df.to_csv(SUMMARY_BY_SIDE_CSV, index=False)

print("Saved pricing CSV:", PRICING_OUTPUT_CSV)
print("Saved pricing parquet:", PRICING_OUTPUT_PARQUET)
print("Saved pricing audit:", PRICING_AUDIT_CSV)
print("Saved summary by side:", SUMMARY_BY_SIDE_CSV)

Saved pricing CSV: C:\Users\patri\vrp_project\data\processed\trade_candidate_pricing_v0_1.csv
Saved pricing parquet: C:\Users\patri\vrp_project\data\processed\trade_candidate_pricing_v0_1.parquet
Saved pricing audit: C:\Users\patri\vrp_project\data\audit\trade_candidate_pricing_audit_v0_1.csv
Saved summary by side: C:\Users\patri\vrp_project\data\audit\trade_candidate_pricing_summary_by_side_v0_1.csv


In [12]:
# ============================================================
# QA: inspect not-priced rows
# ============================================================

not_priced_df = pricing_df[pricing_df["pricing_status"] != "priced"].copy()

print("Not-priced rows:", len(not_priced_df))

display(
    not_priced_df[
        [
            "trade_date",
            "trade_side",
            "selected_expiry_date",
            "actual_dte",
            "spx_close",
            "short_strike_selected",
            "entry_bid",
            "entry_ask",
            "entry_quote_status",
            "expiry_spx_status",
            "pricing_status",
        ]
    ]
    .sort_values(["entry_quote_status", "expiry_spx_status", "trade_date"])
)

Not-priced rows: 23


,trade_date,trade_side,selected_expiry_date,actual_dte,spx_close,short_strike_selected,entry_bid,entry_ask,entry_quote_status,expiry_spx_status,pricing_status
1808,20260520,put,20260626,37.0,7432.97,7430.0,127.8,128.90,mapped,missing_expiry_spx_close,not_priced
1809,20260520,call,20260626,37.0,7432.97,8300.0,0.7,0.85,mapped,missing_expiry_spx_close,not_priced
1826,20260605,put,20260702,27.0,7383.74,7380.0,116.3,120.60,mapped,missing_expiry_spx_close,not_priced
1827,20260609,put,20260702,23.0,7386.65,7385.0,111.1,117.70,mapped,missing_expiry_spx_close,not_priced
1828,20260610,put,20260702,22.0,7266.99,7265.0,123.1,132.60,mapped,missing_expiry_spx_close,not_priced
125,20190329,call,20190503,35.0,2834.40,3100.0,0.0,0.20,zero_bid,mapped,not_priced
195,20190628,call,20190719,21.0,2941.76,3160.0,0.0,1.15,zero_bid,mapped,not_priced
217,20190717,call,20190726,9.0,2984.42,3115.0,0.0,0.10,zero_bid,mapped,not_priced
271,20191024,call,20191101,8.0,3010.29,3115.0,0.0,0.15,zero_bid,mapped,not_priced
337,20191220,call,20200117,28.0,3221.22,3440.0,0.0,0.85,zero_bid,mapped,not_priced


In [13]:
# ============================================================
# QA: put performance by signal tier
# ============================================================

priced_put_df = pricing_df[
    (pricing_df["pricing_status"] == "priced")
    & (pricing_df["trade_side"] == "put")
].copy()

put_tier_summary_df = (
    priced_put_df
    .groupby("signal_tier")
    .agg(
        rows=("trade_date", "count"),
        win_rate=("short_option_win", "mean"),
        expired_otm_rate=("short_option_expired_otm", "mean"),
        avg_entry_credit_points=("entry_credit_points", "mean"),
        median_entry_credit_points=("entry_credit_points", "median"),
        avg_pnl_points=("short_option_pnl_points", "mean"),
        median_pnl_points=("short_option_pnl_points", "median"),
        worst_pnl_points=("short_option_pnl_points", "min"),
        avg_normalized_pnl_pct=("normalized_pnl_pct_notional", "mean"),
        median_normalized_pnl_pct=("normalized_pnl_pct_notional", "median"),
        worst_normalized_pnl_pct=("normalized_pnl_pct_notional", "min"),
        avg_normalized_pnl_dollars=("normalized_pnl_dollars", "mean"),
        median_normalized_pnl_dollars=("normalized_pnl_dollars", "median"),
        worst_normalized_pnl_dollars=("normalized_pnl_dollars", "min"),
    )
    .reset_index()
    .sort_values("signal_tier")
)

display(put_tier_summary_df)

,signal_tier,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,avg_normalized_pnl_pct,median_normalized_pnl_pct,worst_normalized_pnl_pct,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars
0,A_strongest,563,0.836590,0.710480,68.492540,63.1,33.833268,55.7,-657.02,0.008316,0.013095,-0.112316,8316.313704,13095.096845,-112316.486943
1,B_good,239,0.682008,0.573222,48.405439,44.4,5.645230,31.3,-195.31,0.000441,0.007668,-0.067685,440.774728,7668.311185,-67685.067422
2,C_acceptable,272,0.735294,0.658088,60.719853,54.7,8.360294,40.1,-651.52,0.000974,0.009209,-0.139149,973.842615,9209.368587,-139149.196036


In [14]:
# ============================================================
# QA: performance by selected tenor
# ============================================================

tenor_summary_df = (
    pricing_df[pricing_df["pricing_status"] == "priced"]
    .groupby(["trade_side", "selected_tenor"])
    .agg(
        rows=("trade_date", "count"),
        win_rate=("short_option_win", "mean"),
        expired_otm_rate=("short_option_expired_otm", "mean"),
        avg_entry_credit_points=("entry_credit_points", "mean"),
        avg_pnl_points=("short_option_pnl_points", "mean"),
        median_pnl_points=("short_option_pnl_points", "median"),
        worst_pnl_points=("short_option_pnl_points", "min"),
        avg_normalized_pnl_pct=("normalized_pnl_pct_notional", "mean"),
        median_normalized_pnl_pct=("normalized_pnl_pct_notional", "median"),
        worst_normalized_pnl_pct=("normalized_pnl_pct_notional", "min"),
        avg_normalized_pnl_dollars=("normalized_pnl_dollars", "mean"),
        median_normalized_pnl_dollars=("normalized_pnl_dollars", "median"),
    )
    .reset_index()
    .sort_values(["trade_side", "selected_tenor"])
)

display(tenor_summary_df)

,trade_side,selected_tenor,rows,win_rate,expired_otm_rate,avg_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,avg_normalized_pnl_pct,median_normalized_pnl_pct,worst_normalized_pnl_pct,avg_normalized_pnl_dollars,median_normalized_pnl_dollars
0,call,9.0,69,1.000000,1.000000,0.210870,0.210870,0.150,0.05,0.000047,0.000042,0.000007,47.383644,41.818200
1,call,12.0,111,1.000000,1.000000,0.239640,0.239640,0.200,0.05,0.000056,0.000049,0.000007,55.899782,49.351521
2,call,15.0,78,1.000000,1.000000,0.258333,0.258333,0.250,0.05,0.000059,0.000047,0.000008,59.150537,47.355195
3,call,18.0,75,1.000000,1.000000,0.308000,0.308000,0.300,0.05,0.000069,0.000066,0.000011,68.790532,66.145816
4,call,21.0,73,1.000000,1.000000,0.303425,0.303425,0.300,0.05,0.000072,0.000066,0.000012,71.653878,66.132609
5,call,24.0,61,1.000000,1.000000,0.299180,0.299180,0.300,0.05,0.000069,0.000064,0.000017,69.225301,63.823541
6,call,27.0,83,1.000000,1.000000,0.316867,0.316867,0.300,0.05,0.000068,0.000067,0.000015,67.984834,66.629946
7,call,30.0,152,1.000000,1.000000,0.295066,0.295066,0.250,0.05,0.000065,0.000061,0.000011,64.652085,60.837094
8,call,33.0,30,1.000000,1.000000,0.335000,0.335000,0.225,0.05,0.000074,0.000058,0.000011,73.628123,57.678365
9,put,9.0,87,0.747126,0.643678,42.022989,5.307356,28.800,-377.16,0.001205,0.007226,-0.091512,1205.018841,7225.758433


In [15]:
# ============================================================
# QA: performance by signal state
# ============================================================

signal_state_summary_df = (
    pricing_df[pricing_df["pricing_status"] == "priced"]
    .groupby(["signal_state", "trade_side"])
    .agg(
        rows=("trade_date", "count"),
        win_rate=("short_option_win", "mean"),
        expired_otm_rate=("short_option_expired_otm", "mean"),
        avg_pnl_points=("short_option_pnl_points", "mean"),
        median_pnl_points=("short_option_pnl_points", "median"),
        worst_pnl_points=("short_option_pnl_points", "min"),
        avg_normalized_pnl_pct=("normalized_pnl_pct_notional", "mean"),
        median_normalized_pnl_pct=("normalized_pnl_pct_notional", "median"),
        worst_normalized_pnl_pct=("normalized_pnl_pct_notional", "min"),
        avg_normalized_pnl_dollars=("normalized_pnl_dollars", "mean"),
        median_normalized_pnl_dollars=("normalized_pnl_dollars", "median"),
    )
    .reset_index()
    .sort_values(["signal_state", "trade_side"])
)

display(signal_state_summary_df)

,signal_state,trade_side,rows,win_rate,expired_otm_rate,avg_pnl_points,median_pnl_points,worst_pnl_points,avg_normalized_pnl_pct,median_normalized_pnl_pct,worst_normalized_pnl_pct,avg_normalized_pnl_dollars,median_normalized_pnl_dollars
0,both_put_and_call,call,570,1.000000,1.000000,0.262982,0.25,0.05,0.000058,0.000052,0.000007,58.159534,51.585391
1,both_put_and_call,put,583,0.780446,0.667238,22.310720,38.60,-441.66,0.004700,0.009728,-0.112644,4700.198052,9727.895867
2,call_only,call,162,1.000000,1.000000,0.346296,0.35,0.05,0.000082,0.000073,0.000012,81.658166,73.226010
3,put_only,put,491,0.771894,0.665988,19.682668,56.00,-657.02,0.004709,0.013183,-0.139149,4708.960290,13182.995487


In [16]:
# ============================================================
# QA: performance by calendar year
# ============================================================

def trade_year_from_yyyymmdd(date_int):
    return int(str(int(date_int))[:4])

pricing_df["trade_year"] = pricing_df["trade_date"].apply(trade_year_from_yyyymmdd)

year_summary_df = (
    pricing_df[pricing_df["pricing_status"] == "priced"]
    .groupby(["trade_side", "trade_year"])
    .agg(
        rows=("trade_date", "count"),
        win_rate=("short_option_win", "mean"),
        expired_otm_rate=("short_option_expired_otm", "mean"),
        avg_pnl_points=("short_option_pnl_points", "mean"),
        median_pnl_points=("short_option_pnl_points", "median"),
        worst_pnl_points=("short_option_pnl_points", "min"),
        avg_normalized_pnl_pct=("normalized_pnl_pct_notional", "mean"),
        median_normalized_pnl_pct=("normalized_pnl_pct_notional", "median"),
        worst_normalized_pnl_pct=("normalized_pnl_pct_notional", "min"),
        avg_normalized_pnl_dollars=("normalized_pnl_dollars", "mean"),
        median_normalized_pnl_dollars=("normalized_pnl_dollars", "median"),
    )
    .reset_index()
    .sort_values(["trade_side", "trade_year"])
)

display(year_summary_df)

,trade_side,trade_year,rows,win_rate,expired_otm_rate,avg_pnl_points,median_pnl_points,worst_pnl_points,avg_normalized_pnl_pct,median_normalized_pnl_pct,worst_normalized_pnl_pct,avg_normalized_pnl_dollars,median_normalized_pnl_dollars
0,call,2018,24,1.000000,1.000000,0.279167,0.250,0.15,0.000096,0.000087,0.000051,96.077191,86.904749
1,call,2019,129,1.000000,1.000000,0.144574,0.100,0.05,0.000049,0.000037,0.000016,48.976938,36.608447
2,call,2020,93,1.000000,1.000000,0.257527,0.250,0.05,0.000076,0.000068,0.000015,76.139814,68.241498
3,call,2021,109,1.000000,1.000000,0.150459,0.150,0.05,0.000036,0.000033,0.000011,35.509697,32.848058
4,call,2022,23,1.000000,1.000000,0.386957,0.350,0.10,0.000094,0.000085,0.000021,93.921861,84.900557
5,call,2023,103,1.000000,1.000000,0.385437,0.350,0.05,0.000088,0.000079,0.000012,88.011496,78.553009
6,call,2024,122,1.000000,1.000000,0.406557,0.400,0.10,0.000076,0.000074,0.000017,75.593146,73.503354
7,call,2025,102,1.000000,1.000000,0.314706,0.275,0.05,0.000049,0.000044,0.000008,48.901978,43.612575
8,call,2026,27,1.000000,1.000000,0.370370,0.350,0.05,0.000051,0.000050,0.000007,50.680278,50.397275
9,put,2018,29,0.413793,0.379310,-31.528276,-13.130,-195.31,-0.010537,-0.004498,-0.067685,-10536.711523,-4497.545703


In [17]:
# ============================================================
# QA: tails and unusual rows
# ============================================================

print("Worst 25 put losses:")
display(
    pricing_df[
        (pricing_df["pricing_status"] == "priced")
        & (pricing_df["trade_side"] == "put")
    ]
    .sort_values("normalized_pnl_dollars")
    [
        [
            "trade_date",
            "selected_expiry_date",
            "actual_dte",
            "selected_tenor",
            "signal_tier",
            "signal_state",
            "spx_close",
            "expiry_spx_close",
            "short_strike_selected",
            "entry_bid",
            "expiry_intrinsic_value",
            "short_option_pnl_points",
            "normalized_pnl_dollars",
            "normalized_pnl_pct_notional",
            "signal_primary_vrp",
            "signal_blended_z",
            "spx_rsi_14",
        ]
    ]
    .head(25)
)

print("\nLargest 25 call credits:")
display(
    pricing_df[
        (pricing_df["pricing_status"] == "priced")
        & (pricing_df["trade_side"] == "call")
    ]
    .sort_values("entry_credit_points", ascending=False)
    [
        [
            "trade_date",
            "selected_expiry_date",
            "actual_dte",
            "selected_tenor",
            "signal_state",
            "spx_close",
            "expiry_spx_close",
            "short_strike_selected",
            "entry_bid",
            "entry_ask",
            "entry_spread_pct_mid",
            "expiry_intrinsic_value",
            "short_option_pnl_points",
            "normalized_pnl_dollars",
            "normalized_pnl_pct_notional",
            "signal_primary_vrp",
            "signal_blended_z",
            "spx_rsi_14",
        ]
    ]
    .head(25)
)

Worst 25 put losses:


,trade_date,selected_expiry_date,actual_dte,selected_tenor,signal_tier,signal_state,spx_close,expiry_spx_close,short_strike_selected,entry_bid,expiry_intrinsic_value,short_option_pnl_points,normalized_pnl_dollars,normalized_pnl_pct_notional,signal_primary_vrp,signal_blended_z,spx_rsi_14
390,20200224,20200313,18.0,18.0,C_acceptable,put_only,3225.89,2711.02,3225.0,65.1,513.98,-448.88,-139149.196036,-0.139149,0.944174,0.231537,36.699503
386,20200219,20200306,16.0,15.0,C_acceptable,both_put_and_call,3386.15,2972.37,3385.0,31.2,412.63,-381.43,-112644.153390,-0.112644,0.962067,0.306802,65.614116
1486,20250303,20250404,32.0,30.0,A_strongest,put_only,5849.72,5074.08,5845.0,113.9,770.92,-657.02,-112316.486943,-0.112316,0.995026,0.957791,37.160997
1488,20250305,20250404,30.0,30.0,C_acceptable,put_only,5842.63,5074.08,5840.0,114.4,765.92,-651.52,-111511.425505,-0.111511,0.773114,0.437754,39.448300
925,20220912,20220930,18.0,15.0,A_strongest,put_only,4110.41,3585.62,4110.0,78.5,524.38,-445.88,-108475.796818,-0.108476,0.833395,1.124479,54.578509
1487,20250304,20250404,31.0,30.0,A_strongest,put_only,5778.15,5074.08,5775.0,118.6,700.92,-582.32,-100779.661310,-0.100780,0.991487,0.930650,33.223807
1490,20250307,20250404,28.0,30.0,C_acceptable,put_only,5770.20,5074.08,5770.0,116.5,695.92,-579.42,-100415.930124,-0.100416,0.756667,0.371519,36.833340
389,20200221,20200306,14.0,12.0,A_strongest,put_only,3337.75,2972.37,3335.0,37.1,362.63,-325.53,-97529.773051,-0.097530,1.302942,0.791398,53.661884
1489,20250306,20250404,29.0,30.0,A_strongest,put_only,5738.52,5074.08,5730.0,124.6,655.92,-531.32,-92588.332880,-0.092588,0.916220,0.737497,33.946232
892,20220606,20220617,11.0,9.0,A_strongest,put_only,4121.43,3674.84,4120.0,68.0,445.16,-377.16,-91511.926686,-0.091512,0.815093,1.301274,50.962658



Largest 25 call credits:


,trade_date,selected_expiry_date,actual_dte,selected_tenor,signal_state,spx_close,expiry_spx_close,short_strike_selected,entry_bid,entry_ask,entry_spread_pct_mid,expiry_intrinsic_value,short_option_pnl_points,normalized_pnl_dollars,normalized_pnl_pct_notional,signal_primary_vrp,signal_blended_z,spx_rsi_14
1175,20231219,20240119,31.0,33.0,call_only,4768.37,4839.81,5125.0,1.05,1.15,0.090909,0.0,1.05,220.201033,0.000220,0.991986,1.411660,82.184415
415,20200605,20200619,14.0,15.0,call_only,3193.93,3097.74,3480.0,0.90,1.30,0.363636,0.0,0.90,281.784510,0.000282,0.449688,0.492862,72.478105
1172,20231214,20240112,29.0,30.0,call_only,4719.55,4783.83,5050.0,0.85,0.95,0.111111,0.0,0.85,180.101916,0.000180,0.953963,1.413348,79.046948
417,20200608,20200626,18.0,18.0,both_put_and_call,3232.39,3009.05,3600.0,0.85,1.00,0.162162,0.0,0.85,262.963318,0.000263,0.770507,1.072434,74.481355
941,20221201,20221223,22.0,21.0,call_only,4076.57,3844.82,4500.0,0.85,1.00,0.162162,0.0,0.85,208.508624,0.000209,0.344387,0.639951,63.094097
1305,20240612,20240712,30.0,27.0,both_put_and_call,5421.03,5615.35,5800.0,0.85,0.95,0.111111,0.0,0.85,156.796771,0.000157,0.720114,0.554243,71.716083
1051,20230615,20230630,15.0,12.0,call_only,4425.84,4450.38,4675.0,0.80,0.95,0.171429,0.0,0.80,180.756647,0.000181,0.598602,0.234003,76.358207
1344,20240715,20240816,32.0,30.0,both_put_and_call,5631.22,5554.25,6070.0,0.80,0.90,0.117647,0.0,0.80,142.065130,0.000142,1.163349,1.429274,73.584174
1178,20231221,20240119,29.0,30.0,call_only,4746.75,4839.81,5120.0,0.80,0.90,0.117647,0.0,0.80,168.536367,0.000169,0.638341,0.471175,70.425156
903,20220721,20220819,29.0,27.0,call_only,3998.95,4228.48,4520.0,0.80,0.85,0.060606,0.0,0.80,200.052514,0.000200,0.461582,0.450829,59.511990


In [18]:
# ============================================================
# Save QA summary files
# ============================================================

PUT_TIER_SUMMARY_CSV = AUDIT_DIR / "trade_candidate_pricing_put_tier_summary_v0_1.csv"
TENOR_SUMMARY_CSV = AUDIT_DIR / "trade_candidate_pricing_tenor_summary_v0_1.csv"
SIGNAL_STATE_SUMMARY_CSV = AUDIT_DIR / "trade_candidate_pricing_signal_state_summary_v0_1.csv"
YEAR_SUMMARY_CSV = AUDIT_DIR / "trade_candidate_pricing_year_summary_v0_1.csv"

put_tier_summary_df.to_csv(PUT_TIER_SUMMARY_CSV, index=False)
tenor_summary_df.to_csv(TENOR_SUMMARY_CSV, index=False)
signal_state_summary_df.to_csv(SIGNAL_STATE_SUMMARY_CSV, index=False)
year_summary_df.to_csv(YEAR_SUMMARY_CSV, index=False)

# Save final pricing file again, now with trade_year included.
pricing_df.to_csv(PRICING_OUTPUT_CSV, index=False)
pricing_df.to_parquet(PRICING_OUTPUT_PARQUET, index=False)

print("Saved put tier summary:", PUT_TIER_SUMMARY_CSV)
print("Saved tenor summary:", TENOR_SUMMARY_CSV)
print("Saved signal-state summary:", SIGNAL_STATE_SUMMARY_CSV)
print("Saved year summary:", YEAR_SUMMARY_CSV)
print("Re-saved pricing output:", PRICING_OUTPUT_CSV)
print("Re-saved pricing output:", PRICING_OUTPUT_PARQUET)

Saved put tier summary: C:\Users\patri\vrp_project\data\audit\trade_candidate_pricing_put_tier_summary_v0_1.csv
Saved tenor summary: C:\Users\patri\vrp_project\data\audit\trade_candidate_pricing_tenor_summary_v0_1.csv
Saved signal-state summary: C:\Users\patri\vrp_project\data\audit\trade_candidate_pricing_signal_state_summary_v0_1.csv
Saved year summary: C:\Users\patri\vrp_project\data\audit\trade_candidate_pricing_year_summary_v0_1.csv
Re-saved pricing output: C:\Users\patri\vrp_project\data\processed\trade_candidate_pricing_v0_1.csv
Re-saved pricing output: C:\Users\patri\vrp_project\data\processed\trade_candidate_pricing_v0_1.parquet
